# Basic AequilibraE use

In this example, we grab a pre-existing model and do the following:

* Add new fields to the network
* Re-compute fields in the network
* Visualize the network
* Perform Skimming
* Perform Assignment


## Running on Google Colab

Press here to open this notebook in Google Colab <a href="https://colab.research.google.com/github/outerl/AequilibraE-demo/blob/main/basic_aequilibrae_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

Once in Colab, uncomment the cell below and run it.

**It will restart your Python environment to load all packages correctly**, so it is not possible just run all cells

In [1]:
# !apt-get update && apt-get install libsqlite3-mod-spatialite
# !apt-get install -y libspatialite-dev
# !pip install numpy --upgrade
# !pip install aequilibrae matplotlib
# exit()

# Before we begin
Let's make a copy of the model so we can make the changes we want without overwriting the original model

If you have are running this notebook on Google Colab and want to save your model into your Google Drive, you should:

1. Un-comment and run the two cells below
2. Accept the terms and conditions when asking to connect Colab to Google Drive.
3. Skip the following cell, where we copy the model around

In [2]:
# #Just in case you wabt to use Google colab
# from google.colab import drive
# drive.mount("/content/gdrive")

In [3]:
# !wget https://github.com/outerl/AequilibraE-demo/releases/download/Freeworld/LongAn_base_model.zip
# import zipfile
# with zipfile.ZipFile('LongAn_base_model.zip',"r") as zip_ref:
#     zip_ref.extractall(".")

# # You probably want to move this into your google drive, but we will leave in iun the temporary folder of the VM running the model
# model_path = "./LongAn_base_model"

In [4]:
model_path = r"C:\Users\Pedro\OneDrive\poli\LongAn_base_model"

If not using Colab, you will need to download the [MODEL](https://github.com/outerl/AequilibraE-demo/releases/download/Freeworld/LongAn_base_model.zip), unzip it and set the folders below accordingly

## Let's do some modeling

In [5]:
# Imports
from pathlib import Path
from aequilibrae import Project

In [6]:
# We open a project
project = Project.from_path(Path(model_path))

## Model stats

In [7]:
print(f"Links: {project.network.count_links():,}")
print(f"Nodes: {project.network.count_nodes():,}")
print(f"Zones: {project.network.count_centroids():,}")

Links: 253,609
Nodes: 251,335
Zones: 698


Let's see which fields we have in our links layer

In [8]:
project.network.links.fields.all_fields()

['a_node',
 'b_node',
 'capacity',
 'direction',
 'distance',
 'ff_ttime',
 'geometry',
 'lanes',
 'link_id',
 'link_type',
 'modes',
 'name',
 'osm_id',
 'speed']

We are missing travel time. Let's add it

In [9]:
fields = project.network.links.fields
if "ff_ttime_ab" not in project.network.links.fields.all_fields():
    fields.add("ff_ttime_ab", description="AB Free flow travel time")
    fields.add("ff_ttime_ba", description="BA Free flow travel time")
    fields.save()

In [10]:
project.network.links.fields.all_fields()

['a_node',
 'b_node',
 'capacity',
 'direction',
 'distance',
 'ff_ttime',
 'geometry',
 'lanes',
 'link_id',
 'link_type',
 'modes',
 'name',
 'osm_id',
 'speed']

#### Let's use some SQL to compute these fields

Speeds are in KM/h and distances are ALWAYS in meters in AequilibraE


In [11]:
%%time
sql = """Update links set ff_ttime_ab=(distance/1000)/speed_ab * 60, 
                          ff_ttime_ba=(distance/1000)/speed_ab * 60"""
with project.db_connection_spatial as conn:
    conn.execute(sql)

CPU times: total: 3.83 s
Wall time: 6.48 s


In [12]:
links = project.network.links.data
nodes = project.network.nodes.data
zones = project.zoning.data

In [13]:
links.head(3)

,ogc_fid,link_id,a_node,b_node,direction,distance,modes,link_type,name,speed_ab,speed_ba,capacity_ab,capacity_ba,osm_id,lanes_ab,lanes_ba,ff_ttime_ab,ff_ttime_ba,geometry
0,1036980,3522,809702,809698,1,152.709150,ct,trunk,Quốc lộ 1,50.0,NaN,2400.0,NaN,908323334.0,2,NaN,0.183251,0.183251,"LINESTRING (109.06511 14.2137, 109.06528 14.21..."
1,1036981,3523,809698,809761,1,91.115626,ct,trunk,Quốc lộ 1,50.0,NaN,2400.0,NaN,908323334.0,2,NaN,0.109339,0.109339,"LINESTRING (109.06567 14.21497, 109.06586 14.2..."
2,1036982,3524,809761,809753,1,1095.520931,ct,trunk,Quốc lộ 1,50.0,NaN,2400.0,NaN,908323334.0,2,NaN,1.314625,1.314625,"LINESTRING (109.06602 14.21571, 109.06618 14.2..."


In [14]:
nodes.head(3)

,ogc_fid,node_id,is_centroid,modes,link_types,osm_id,geometry
0,808665,808665,0,ct,a,6.514324e+09,POINT (109.05933 14.69109)
1,808678,808678,0,ct,a,5.962980e+09,POINT (109.05379 14.60347)
2,808714,808714,0,ct,a,6.699966e+09,POINT (109.05761 14.61647)


In [15]:
zones.tail(3)

,ogc_fid,zone_id,area,name,population,employment,province,district,geometry
695,696,696,None,None,None,None,Long An,Can Giuoc,"MULTIPOLYGON (((106.6572 10.62722, 106.65614 1..."
696,697,697,None,None,None,None,Long An,Can Giuoc,"MULTIPOLYGON (((106.66349 10.61996, 106.66402 ..."
697,698,698,None,None,None,None,Long An,Can Giuoc,"MULTIPOLYGON (((106.65872 10.61546, 106.65766 ..."


## Visualize the network

In [23]:

links_in_zone = links[links.intersects(zones[zones.zone_id == 698])]
nodes_in_zone = nodes[nodes.intersects(zones[zones.zone_id == 698])]

m = links_in_zone.explore(
    column="link_type",  # field to color by
    categorical=True,  # treat values as categories
    cmap="Set1",  # color palette for categories
    legend=True,  # force legend
    legend_kwds={"caption": "Link type"},
    tooltip=["link_id", "link_type"],
    style_kwds={"weight": 3},
)
#
# m = nodes_in_zone.explore(
#     column="is_centroid",  # field to color by
#     categorical=True,  # treat values as categories
#     cmap="Set1",  # color palette for categories
#     legend=True,  # force legend
#     legend_kwds={"caption": "is centroid"},
#     tooltip=["node_id", "is_centroid"],
#     style_kwds={"weight": 3},
#     m=m
# )


m

C:\Users\Pedro\AppData\Local\Temp\ipykernel_36612\2849801096.py:1: UserWarning: The indices of the left and right GeoSeries' are not equal, and therefore they will be aligned (reordering and/or introducing missing values) before executing the operation. If this alignment is the desired behaviour, you can silence this warning by passing 'align=True'. If you don't want alignment and protect yourself of accidentally aligning, you can pass 'align=False'.
  links_in_zone = links[links.intersects(zones[zones.zone_id == 698])]
C:\Users\Pedro\AppData\Local\Temp\ipykernel_36612\2849801096.py:2: UserWarning: The indices of the left and right GeoSeries' are not equal, and therefore they will be aligned (reordering and/or introducing missing values) before executing the operation. If this alignment is the desired behaviour, you can silence this warning by passing 'align=True'. If you don't want alignment and protect yourself of accidentally aligning, you can pass 'align=False'.
  nodes_in_zone = n

In [24]:
links

,ogc_fid,link_id,a_node,b_node,direction,distance,modes,link_type,name,speed_ab,speed_ba,capacity_ab,capacity_ba,osm_id,lanes_ab,lanes_ba,ff_ttime_ab,ff_ttime_ba,geometry
0,1036980,3522,809702,809698,1,152.709150,ct,trunk,Quốc lộ 1,50.0,NaN,2400.0,NaN,908323334.0,2,NaN,0.183251,0.183251,"LINESTRING (109.06511 14.2137, 109.06528 14.21..."
1,1036981,3523,809698,809761,1,91.115626,ct,trunk,Quốc lộ 1,50.0,NaN,2400.0,NaN,908323334.0,2,NaN,0.109339,0.109339,"LINESTRING (109.06567 14.21497, 109.06586 14.2..."
2,1036982,3524,809761,809753,1,1095.520931,ct,trunk,Quốc lộ 1,50.0,NaN,2400.0,NaN,908323334.0,2,NaN,1.314625,1.314625,"LINESTRING (109.06602 14.21571, 109.06618 14.2..."
3,1036983,3525,809753,809762,1,363.367722,ct,trunk,Quốc lộ 1,50.0,NaN,2400.0,NaN,908323334.0,2,NaN,0.436041,0.436041,"LINESTRING (109.06891 14.22507, 109.06895 14.2..."
4,1036984,3526,809762,809763,1,90.227685,ct,trunk,Quốc lộ 1,50.0,NaN,2400.0,NaN,908323334.0,2,NaN,0.108273,0.108273,"LINESTRING (109.06983 14.22823, 109.07003 14.2..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253604,9307928,8274470,7322762,7323917,1,2839.309367,ct,primary,"DT 107, Mường Giàng",40.0,0.0,1800.0,0.0,633175178.0,2,NaN,4.258964,4.258964,"LINESTRING (103.63909 21.63422, 103.63913 21.6..."
253605,9307929,8274471,7538546,7553792,0,62856.887881,ct,trunk,Quốc lộ 12,50.0,50.0,2400.0,2400.0,692586057.0,2,2.0,75.428265,75.428265,"LINESTRING (103.22644 22.48597, 103.22647 22.4..."
253606,9307930,8274472,7539088,7271368,1,2405.262379,ct,trunk,Quốc lộ 4D,50.0,0.0,2400.0,0.0,530008153.0,2,NaN,2.886315,2.886315,"LINESTRING (103.27999 22.53498, 103.28008 22.5..."
253607,9307931,8274473,7570997,7572318,1,1965.186827,ct,primary,None,40.0,0.0,1800.0,0.0,891758193.0,2,NaN,2.947780,2.947780,"LINESTRING (103.23369 21.52395, 103.23354 21.5..."


## Computes Graph



In [ ]:
%%time
project.network.build_graphs(modes=["c"])

In [ ]:
graph = project.network.graphs["c"]
graph.set_graph("ff_ttime")
graph.set_skimming(["distance", "ff_ttime"])

## Skimming

In [ ]:
%%time
skimmer = graph.compute_skims()

In [ ]:
skimmer.results.skims.distance

In [ ]:
skimmer.results.skims.ff_ttime

We can save this matrix to disk and add to the model

In [ ]:
skimmer.save_to_project("skim_matrix")

## Matrices

In [ ]:
project.matrices.list()

In [ ]:
mat = project.matrices.get_matrix("base_demand")
mat.names

In [ ]:
project.matrices.update_database()
project.matrices.list()

## Assignment

Let's assign only a few of the matrices we exported

In [ ]:
demand_cars = project.matrices.get_matrix("base_demand")
demand_cars.computational_view(['red_cars_AM', 'blue_cars_AM'])

In [ ]:
demand_trucks = project.matrices.get_matrix("base_demand")
demand_trucks.computational_view(['Trucks_AM'])

In [ ]:
from aequilibrae.paths import TrafficAssignment, TrafficClass

In [ ]:

# Create the assignment class
car_class = TrafficClass(name="car", graph=graph, matrix=demand_cars)
truck_class = TrafficClass(name="truck", graph=graph, matrix=demand_trucks)

truck_class.set_pce(1.8)
# truck_class.set_select_links()
# truck_class.set_vot()
# truck_class.set_fixed_cost()

In [ ]:

assig = TrafficAssignment()

# The first thing to do is to add at list of traffic classes to be assigned
assig.add_class(car_class)
assig.add_class(truck_class)

# We set these parameters only after adding one class to the assignment
assig.set_vdf("BPR")  # This is not case-sensitive

# Then we set the volume delay function
assig.set_vdf_parameters({"alpha": 0.15, "beta": 4.0})  # And its parameters

assig.set_capacity_field("capacity")  # The capacity and free flow travel times as they exist in the graph
assig.set_time_field("ff_ttime")

# And the algorithm we want to use to assign
assig.set_algorithm("bfw")

# Since I haven't checked the parameters file, let's make sure convergence criteria is good
assig.max_iter = 5
assig.rgap_target = 0.0001

assig.execute()  # we then execute the assignment

In [ ]:
assig.report()

In [ ]:
assig.results().max()

In [ ]:
assig.save_results("tutorial")